In [ ]:
import sys
sys.path.append("../models")
from SVD import SVDModel

In [2]:
import pandas as pd

movies = pd.read_csv('../data/sample/movies.csv')
ratings = pd.read_csv('../data/sample/ratings.csv')
tags = pd.read_csv('../data/sample/tags.csv')

In [33]:
import torch
from SVD import SVDModel
from torch.utils.data import DataLoader, TensorDataset


num_users = ratings['userId'].nunique()
num_items = ratings['movieId'].nunique()
global_mean = ratings['rating'].mean()

# Map userId và movieId sang index liên tục
user_to_idx = {uid: i for i, uid in enumerate(ratings['userId'].unique())}
item_to_idx = {mid: i for i, mid in enumerate(ratings['movieId'].unique())}


user_ids = torch.tensor([user_to_idx[uid] for uid in ratings['userId']], dtype=torch.long)
item_ids = torch.tensor([item_to_idx[mid] for mid in ratings['movieId']], dtype=torch.long)
rating_tensor = torch.tensor(ratings['rating'].values, dtype=torch.float32)

dataset = TensorDataset(user_ids, item_ids, rating_tensor)
train_loader = DataLoader(dataset, batch_size=256, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = SVDModel(num_users, num_items, embedding_dim=64, global_mean=global_mean).to(device)
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    
    # Vòng lặp chạy qua từng batch nhỏ
    for batch_user, batch_item, batch_rating in train_loader:
        batch_user = batch_user.to(device)
        batch_item = batch_item.to(device)
        batch_rating = batch_rating.to(device)
        
        optimizer.zero_grad()
        preds = model(batch_user, batch_item)
        loss = criterion(preds, batch_rating)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch_user.size(0)
        
    epoch_loss = total_loss / len(dataset)
    print(f"Epoch {epoch+1}/{epochs} - Loss trung bình: {epoch_loss:.4f}")

Epoch 1/10 - Loss trung bình: 0.1829
Epoch 2/10 - Loss trung bình: 0.1396
Epoch 3/10 - Loss trung bình: 0.1030
Epoch 4/10 - Loss trung bình: 0.0726
Epoch 5/10 - Loss trung bình: 0.0481
Epoch 6/10 - Loss trung bình: 0.0295
Epoch 7/10 - Loss trung bình: 0.0168
Epoch 8/10 - Loss trung bình: 0.0098
Epoch 9/10 - Loss trung bình: 0.0077
Epoch 10/10 - Loss trung bình: 0.0089


In [35]:
import torch
import pandas as pd


def recommend_movies(model, user_id, ratings, movies, user_to_idx, item_to_idx, top_k=10, device='cpu'):
    model.eval()
    with torch.no_grad():
        # Kiểm tra user_id có trong dataset không
        if user_id not in user_to_idx:
            raise ValueError(f"User {user_id} not found in training data")
        
        # Lấy index user
        u_idx = user_to_idx[user_id]
        num_items = len(item_to_idx)
        
        # Tạo tensor tất cả item cho user này
        user_tensor = torch.tensor([u_idx] * num_items, dtype=torch.long).to(device)
        item_tensor = torch.tensor(list(range(num_items)), dtype=torch.long).to(device)
        
        # Dự đoán rating
        preds = model(user_tensor, item_tensor)
        
        # Chuyển về CPU và pandas Series
        preds = preds.cpu().numpy()
        item_idx_to_id = {idx: mid for mid, idx in item_to_idx.items()}
        pred_series = pd.Series(preds, index=[item_idx_to_id[i] for i in range(num_items)])
        
        # Loại bỏ những phim user đã đánh giá
        watched_items = ratings[ratings['userId'] == user_id]['movieId'].tolist()
        pred_series = pred_series.drop(watched_items, errors='ignore')
        
        # Lấy top_k phim dự đoán rating cao nhất
        top_movies_ids = pred_series.sort_values(ascending=False).head(top_k).index.tolist()
        
        # Lấy tên phim
        top_movies = movies[movies['movieId'].isin(top_movies_ids)][['movieId','title']]
        top_movies = top_movies.set_index('movieId').loc[top_movies_ids]  # giữ thứ tự top k
        
        return top_movies

top_movies = recommend_movies(model, user_id=101, ratings=ratings, movies=movies,
                              user_to_idx=user_to_idx, item_to_idx=item_to_idx,
                              top_k=10, device=device)

print(top_movies)

                             title
movieId                           
4                   Sabrina (1995)
9                Braveheart (1995)
17               The Matrix (1999)
11                Apollo 13 (1995)
20          The Dark Knight (2008)
24                 Parasite (2019)
15                    Alien (1979)
8        Usual Suspects The (1995)
22             Interstellar (2014)
18               Fight Club (1999)


In [36]:
import os
import torch

artifact_path = '../artifacts'
os.makedirs(artifact_path, exist_ok=True) 

torch.save(model.state_dict(), f'{artifact_path}/svd_model.pth')
print("Đã lưu model thành công!")

Đã lưu model thành công!
